In [85]:
import os
import re
import time
import shutil
import pyodbc
import fnmatch
import numpy as np
import pandas as pd
import win32com.client
from math import floor
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

In [42]:
# Generate random iri
def generate_parts(target_values, num_parts, tolerance):
    parts_list = []
    for target_value in target_values:
        total_sum = target_value * num_parts
        while True:
            # Generate random parts
            parts = np.random.uniform(low=total_sum / num_parts * 0.9, high=total_sum / num_parts * 1.1, size=num_parts)
            # Ensure the sum is correct
            if np.abs(np.sum(parts) - total_sum) < tolerance:
                parts_list.append(parts)
                break
            
    return parts_list

# Find all relevant CSV files and process them
def process_csv_files(path):
    all_iri_dataframes = [] # empty list
    all_rutting_dataframes = [] # empty list
    
    for root, dirs, files in os.walk(path):
        # Find files
        iri_files = [f for f in files if f.endswith('.csv') and 'xw_iri_qgis' in f]
        rutting_files = [f for f in files if f.endswith('.csv') and 'xw_rutting' in f]

        # Process 'xw_iri_qgis' files
        for filename in iri_files:
            file_path = os.path.join(root, filename)
            iri_df = pd.read_csv(file_path, delimiter=';')
            iri_df.columns = iri_df.columns.str.strip()
            survey_code = filename.split('_')[3].split('.')[0]
            iri_df['survey_code'] = survey_code
            iri_df['iri'] = (iri_df['iri left (m/km)'] + iri_df['iri right (m/km)']) / 2
            iri_df.drop(columns=['geometry'], errors='ignore', inplace=True)

            # Generate random values for iri_lane
            target_values = iri_df['iri']
            num_parts = 4
            tolerance = 0.3
            parts_list = generate_parts(target_values, num_parts, tolerance)

            # Expand DataFrame by repeating the rows
            iri_df = iri_df.loc[iri_df.index.repeat(num_parts)].reset_index(drop=True)
            iri_df['iri_lane'] = np.concatenate(parts_list)

            increment = 5 if fnmatch.fnmatch(filename, '*xw_iri_qgis*') else 5
            iri_df['event_start'] = range(0, len(iri_df) * increment, increment)
            iri_df['event_end'] = iri_df['event_start'] + increment

            # Append the processed IRI DataFrame to the list
            all_iri_dataframes.append(iri_df)

        # Process 'xw_rutting' files
        for filename in rutting_files:
            file_path = os.path.join(root, filename)
            rut_df = pd.read_csv(file_path, delimiter=';')
            rut_df.columns = rut_df.columns.str.strip()
            if 'Unnamed: 5' in rut_df.columns:
                rut_df.drop(columns=['Unnamed: 5'], inplace=True, errors='ignore')
            else:
                pass
            increment = 5 if fnmatch.fnmatch(filename, '*xw_rutting*') else 5
            rut_df['event_start'] = range(0, len(rut_df) * increment, increment)
            rut_df['event_end'] = rut_df['event_start'] + increment
            rut_df['rut_chainage'] = rut_df['event_start']
            survey_code = filename.split('_')[2].split('.')[0]
            rut_df['survey_code'] = survey_code
            if 'qgis_shape' in rut_df.columns:
                rut_df['rut_point_x'] = rut_df['qgis_shape'].apply(lambda x: float(x.split('(')[1].split(')')[0].split(',')[0].split(' ')[1]))
                rut_df['rut_point_y'] = rut_df['qgis_shape'].apply(lambda x: float(x.split('(')[1].split(')')[0].split(',')[0].split(' ')[0]))
                
                # Apply interpolation with a limit to avoid interpolating across large gaps
                rut_df['rut_point_x'] = rut_df['rut_point_x'].interpolate(method='linear', limit_direction='both')
                rut_df['rut_point_y'] = rut_df['rut_point_y'].interpolate(method='linear', limit_direction='both')

                # Forward/backward fill to close gaps
                rut_df['rut_point_x'].fillna(method='ffill', inplace=True)
                rut_df['rut_point_x'].fillna(method='bfill', inplace=True)
                rut_df['rut_point_y'].fillna(method='ffill', inplace=True)
                rut_df['rut_point_y'].fillna(method='bfill', inplace=True)

                # Replace remaining NaN values with 0 (optional)
                rut_df['rut_point_x'].fillna(0, inplace=True)
                rut_df['rut_point_y'].fillna(0, inplace=True)
                
                rut_df.drop(columns=['qgis_shape'], inplace=True)
            else:
                print("not found 'qgis_shape' or it have a white space ! ")
        
            rut_df.rename(columns={'#Date':'Date', 'left rutting height': 'left_rutting', 'right rutting height': 'right_rutting', 'average height': 'avg_rutting'}, inplace=True)

            all_rutting_dataframes.append(rut_df)

    if all_iri_dataframes:
        iri_dataframes = pd.concat(all_iri_dataframes, ignore_index=True)
    else:
        iri_dataframes = pd.DataFrame()

    if all_rutting_dataframes:
        rutting_dataframes = pd.concat(all_rutting_dataframes, ignore_index=True)
    else:
        rutting_dataframes = pd.DataFrame()
        
    print(f"✅ Finished processing: .CSV files.")
    return iri_dataframes, rutting_dataframes

In [81]:
# Perform the left join on xw_rutting and xw_iri_qgis
def left_join_dataframes(df_rutting, df_iri):
    df_merged = pd.merge(df_rutting, df_iri, how='left', on=['event_start', 'event_end', 'survey_code'], suffixes=('_rutting', '_iri'))
    
    # print("Merged DataFrame columns:", df_merged.columns)

    result = df_merged[
        (df_merged['rut_chainage'] >= df_merged['event_start']) &
        (df_merged['rut_chainage'] < df_merged['event_end'])
    ]
    return result

# Perform jpg file and frame number
def get_jpg_filenames(directory):
    jpg_dict = {}
    for root, dirs, files in os.walk(directory):
        jpg_files = [f for f in files if f.endswith('.jpg')]
        if jpg_files:
            folder_name = os.path.basename(os.path.dirname(root))
            jpg_dict[folder_name] = len(jpg_files)
            
    frame_df = pd.DataFrame(list(jpg_dict.items()), columns=['survey_code','frame_num'])
    frame_df['survey_code'] = frame_df['survey_code'].str.replace(
        r'_(\d+)', lambda m: f"RUN{int(m.group(1)):02d}", regex=True
    )
    
    detailed_df = pd.DataFrame(columns=['frame_num', 'survey_code'])
    for index, row in frame_df.iterrows():
        pic_counts = range(1, int(row['frame_num']) + 1)
        temp_df = pd.DataFrame({
            'frame_num': pic_counts,
            'survey_code': row['survey_code'],
        })
        detailed_df = pd.concat([detailed_df, temp_df], ignore_index=True)

    return detailed_df

def add_frame_num_to_joined_df(joined_df, derived_values, frame_numbers):
    joined_df['frame_num_ch'] = pd.NA
    joined_df['frame_num'] = pd.NA
    
    derived_to_frame_mapping = pd.DataFrame({
        'frame_num_ch': derived_values,
        'frame_num': frame_numbers
    })
    
    for i, frame_num_ch in enumerate(derived_values):
        mask = (joined_df['event_start'] <= frame_num_ch) & (joined_df['event_end'] >= frame_num_ch)
        joined_df.loc[mask, 'frame_num_ch'] = frame_num_ch
        joined_df.loc[mask, 'frame_num'] = frame_numbers[i]
        
    return joined_df

def process_fainal_df(output_dir):
    frame_numbers_df = get_jpg_filenames(output_dir)  # This returns a DataFrame
    frame_numbers = frame_numbers_df['frame_num'].astype(int).tolist()
    print(frame_numbers)
    iri_dataframes, rutting_dataframes = process_csv_files(output_dir)

    joined_df = left_join_dataframes(rutting_dataframes, iri_dataframes)

    grouped_df = joined_df.groupby('survey_code').agg(
        max_chainage=('rut_chainage', 'max'),
        min_chainage=('rut_chainage', 'min')
    ).reset_index()

    joined_df = pd.merge(joined_df, grouped_df, on='survey_code', how='left')
    
    max_event_start = joined_df['event_start'].max()

    # Calculate derived values
    derived_values = [floor((max_event_start * num) / max(frame_numbers)) for num in frame_numbers]
    # derived_values = [((max_event_start * num) / max(frame_numbers)) for num in frame_numbers]

    # Add frame numbers to the joined DataFrame
    final_df = add_frame_num_to_joined_df(joined_df, derived_values, frame_numbers)
    
    final_df = final_df.rename(columns={'rut_chainage':'chainage'})
    
    selected_columns = [
        'left_rutting', 'right_rutting', 'avg_rutting', 'event_start', 'event_end', 'survey_code',
        'rut_point_x', 'rut_point_y', 'Date', 'iri left (m/km)', 'iri right (m/km)', 'iri', 'iri_lane', 
        'chainage', 'max_chainage', 'min_chainage', 'frame_num', 'frame_num_ch'
    ] 
    
    selected_columns = [col for col in selected_columns if col in final_df.columns]
    # final_df = final_df[final_df['iri'].notnull()][selected_columns]
    
    return joined_df, derived_values, final_df

In [82]:
# output_dir = r'D:\xenomatix\output'
output_dir = r'E:\xeno1030\output'
joined_df, derived_values, final_df = process_fainal_df(output_dir)

# final_df.to_csv('joinjoin.csv')
joined_df.tail(15)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 22

,Date_rutting,left_rutting,right_rutting,avg_rutting,event_start,event_end,rut_chainage,survey_code,rut_point_x,rut_point_y,Date_iri,iri left (m/km),iri Std left (m/km),iri right (m/km),iri Std right (m/km),worst iri (m/km),iri difference (m/km),"geometry,",iri,iri_lane,max_chainage,min_chainage,frame_num_ch,frame_num
2747,05/11/2024,0.40,0.29,0.34,4520,4525,4520,20241030RUN02,7.010527,100.721082,05/11/2024,2.555524,1.739736,2.609390,1.940678,2.609390,0.053866,LINESTRING(100.72108184786593 7.01052655281898...,2.582457,2.372948,4590,0,4524,899
2748,05/11/2024,1.67,0.17,0.92,4525,4530,4525,20241030RUN02,7.010569,100.721067,05/11/2024,2.555524,1.739736,2.609390,1.940678,2.609390,0.053866,LINESTRING(100.72108184786593 7.01052655281898...,2.582457,2.821623,4590,0,4530,900
2749,05/11/2024,2.32,0.19,1.25,4530,4535,4530,20241030RUN02,7.010612,100.721052,05/11/2024,2.555524,1.739736,2.609390,1.940678,2.609390,0.053866,LINESTRING(100.72108184786593 7.01052655281898...,2.582457,2.553308,4590,0,4535,901
2750,05/11/2024,1.17,0.40,0.78,4535,4540,4535,20241030RUN02,7.010655,100.721037,05/11/2024,2.555524,1.739736,2.609390,1.940678,2.609390,0.053866,LINESTRING(100.72108184786593 7.01052655281898...,2.582457,2.421563,4590,0,4540,902
2751,05/11/2024,0.32,0.14,0.23,4540,4545,4540,20241030RUN02,7.010697,100.721022,05/11/2024,3.050314,2.126427,3.471181,2.459115,3.471181,0.420868,LINESTRING(100.72102243246746 7.01069737961412...,3.260748,3.543385,4590,0,4545,903
2752,05/11/2024,1.33,0.33,0.83,4545,4550,4545,20241030RUN02,7.010740,100.721008,05/11/2024,3.050314,2.126427,3.471181,2.459115,3.471181,0.420868,LINESTRING(100.72102243246746 7.01069737961412...,3.260748,3.006949,4590,0,4550,904
2753,05/11/2024,3.63,2.06,2.84,4550,4555,4550,20241030RUN02,7.010783,100.720993,05/11/2024,3.050314,2.126427,3.471181,2.459115,3.471181,0.420868,LINESTRING(100.72102243246746 7.01069737961412...,3.260748,3.289203,4590,0,4555,905
2754,05/11/2024,4.93,2.73,3.83,4555,4560,4555,20241030RUN02,7.010826,100.720979,05/11/2024,3.050314,2.126427,3.471181,2.459115,3.471181,0.420868,LINESTRING(100.72102243246746 7.01069737961412...,3.260748,3.023314,4590,0,4560,906
2755,05/11/2024,5.62,2.68,4.15,4560,4565,4560,20241030RUN02,7.010869,100.720965,05/11/2024,4.926702,3.828804,6.889028,4.925679,6.889028,1.962326,LINESTRING(100.7209651596047 7.010868932277121...,5.907865,6.343254,4590,0,4565,907
2756,05/11/2024,3.98,1.89,2.94,4565,4570,4565,20241030RUN02,7.010912,100.720951,05/11/2024,4.926702,3.828804,6.889028,4.925679,6.889028,1.962326,LINESTRING(100.7209651596047 7.010868932277121...,5.907865,6.024217,4590,0,4570,908


In [53]:
df = pd.DataFrame(derived_values)

df.head(1)

,0
0,5


In [84]:
final_df.head(10)

,Date_rutting,left_rutting,right_rutting,avg_rutting,event_start,event_end,chainage,survey_code,rut_point_x,rut_point_y,Date_iri,iri left (m/km),iri Std left (m/km),iri right (m/km),iri Std right (m/km),worst iri (m/km),iri difference (m/km),"geometry,",iri,iri_lane,max_chainage,min_chainage,frame_num_ch,frame_num
0,05/11/2024,0.51,3.07,1.79,0,5,0,20241030RUN01,6.948869,100.746381,05/11/2024,3.678751,3.235021,3.163163,2.226583,3.678751,0.515588,LINESTRING(100.74638060721544 6.94886860531095...,3.420957,3.607635,9210,0,5,1
1,05/11/2024,0.63,2.59,1.61,5,10,5,20241030RUN01,6.948907,100.746404,05/11/2024,3.678751,3.235021,3.163163,2.226583,3.678751,0.515588,LINESTRING(100.74638060721544 6.94886860531095...,3.420957,3.090399,9210,0,10,2
2,05/11/2024,2.25,4.01,3.13,10,15,10,20241030RUN01,6.948946,100.746428,05/11/2024,3.678751,3.235021,3.163163,2.226583,3.678751,0.515588,LINESTRING(100.74638060721544 6.94886860531095...,3.420957,3.693618,9210,0,15,3
3,05/11/2024,0.66,2.90,1.78,15,20,15,20241030RUN01,6.948985,100.746451,05/11/2024,3.678751,3.235021,3.163163,2.226583,3.678751,0.515588,LINESTRING(100.74638060721544 6.94886860531095...,3.420957,3.163412,9210,0,20,4
4,05/11/2024,0.18,2.57,1.38,20,25,20,20241030RUN01,6.949024,100.746474,05/11/2024,2.334077,1.851439,2.590045,1.983087,2.590045,0.255968,LINESTRING(100.74647353233989 6.94902378685747...,2.462061,2.482799,9210,0,25,5
5,05/11/2024,0.49,3.06,1.78,25,30,25,20241030RUN01,6.949063,100.746496,05/11/2024,2.334077,1.851439,2.590045,1.983087,2.590045,0.255968,LINESTRING(100.74647353233989 6.94902378685747...,2.462061,2.707459,9210,0,30,6
6,05/11/2024,1.04,2.97,2.00,30,35,30,20241030RUN01,6.949102,100.746519,05/11/2024,2.334077,1.851439,2.590045,1.983087,2.590045,0.255968,LINESTRING(100.74647353233989 6.94902378685747...,2.462061,2.295157,9210,0,35,7
7,05/11/2024,1.00,3.59,2.30,35,40,35,20241030RUN01,6.949141,100.746542,05/11/2024,2.334077,1.851439,2.590045,1.983087,2.590045,0.255968,LINESTRING(100.74647353233989 6.94902378685747...,2.462061,2.328770,9210,0,40,8
8,05/11/2024,0.43,3.31,1.87,40,45,40,20241030RUN01,6.949180,100.746565,05/11/2024,1.582355,1.080961,1.796132,1.298717,1.796132,0.213778,LINESTRING(100.74656493967635 6.94917986616949...,1.689244,1.659075,9210,0,45,9
9,05/11/2024,0.25,4.49,2.37,45,50,45,20241030RUN01,6.949218,100.746589,05/11/2024,1.582355,1.080961,1.796132,1.298717,1.796132,0.213778,LINESTRING(100.74656493967635 6.94917986616949...,1.689244,1.628276,9210,0,50,10


In [57]:
def find_csv_files(start_dir, prefix='log_'):
    csv_files = []
    for dirpath, dirnames, filenames in os.walk(start_dir):
        for filename in fnmatch.filter(filenames, f'{prefix}*.xlsx'):
            csv_files.append(os.path.join(dirpath, filename))
    return csv_files

def main(final_df, output_dir):
    for survey_date in os.listdir(output_dir): # eg. base_dir = r"D:\xenomatixs"
        path = os.path.join(output_dir, survey_date, 'Output')
        mdb = os.path.join(output_dir, survey_date, 'Data')
        
        log_csv_files = find_csv_files(path)
        if log_csv_files:
            log_df = pd.read_excel(log_csv_files[0])
            log_df.rename(columns={'ผิว': 'event_name', 'link_id ระบบ': 'section_id'}, inplace=True)
            log_df.columns = log_df.columns.str.strip()

            folder_names = [name for name in os.listdir(path) if os.path.isdir(os.path.join(path, name))]
            for folder_name in folder_names:
                print(f"🔄 Processing folder: {folder_name}")
                
                # Perform the initial merge and filter rows where frame_num is between numb_start and numb_end
                merged_df = pd.merge(final_df, log_df, how='left', on=['survey_code'], suffixes=('_final_df', '_log_df'))
                merged_df = merged_df[(merged_df['frame_num'] >= merged_df['numb_start']) & 
                                    (merged_df['frame_num'] <= merged_df['numb_end'])]
                
                filtered_df = merged_df[merged_df['survey_code'] == folder_name]
                run_code = re.sub(r'RUN0*(\d+)', r'_\1', folder_name)
                
                # # add filter_df as min_chainage and max_chainage is group by numb_start and numb_end and merge to merged_df
                # filter_df = merged_df.groupby(['numb_start', 'numb_end'], group_keys=False).agg(
                #     min_chainage=('chainage', 'min'),
                #     max_chainage=('chainage', 'max')
                # ).reset_index()
                
                # merged_df = pd.merge(merged_df, filter_df, on=['numb_start', 'numb_end'], how='left')
                # filtered_df = pd.merge(filtered_df, filter_df, on=['numb_start', 'numb_end'], how='left')
    return merged_df, filtered_df

In [87]:
merged_df, filtered_df = main(final_df, output_dir)

🔄 Processing folder: 20241030RUN01
🔄 Processing folder: 20241030RUN02


In [91]:
# merged_df.head(200)